# Neurova: retrain the models in Google Colab

This notebook runs the project's own training scripts, so what happens here is exactly what `training/train_mri.py` and `training/train_sleep.py` do on a laptop. You only need it if you want to retrain. The app already contains trained models.

Run the cells from top to bottom. The MRI training takes about two hours and the sleep training about five minutes. No GPU is needed.

**1. Upload the project.** Zip your `neurova` folder, run this cell and choose the zip.

In [ ]:
from google.colab import files
import zipfile, io, os
uploaded = files.upload()
zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))).extractall(".")
%cd neurova

**2. Install what training needs** (PyTorch is already in Colab).

In [ ]:
!pip -q install -r requirements.txt -r training/requirements.txt

**3. Download the public datasets.** The Brain Tumor MRI Dataset (CC BY 4.0) and 20 nights of Sleep-EDF from PhysioNet. About 1.2 GB in total.

In [ ]:
!python training/get_data.py --sleep

**4. Train the three MRI models and compare them.** A small CNN from scratch, EfficientNet-B0, and EfficientNet-B0 with fingerprint-scrub augmentation. The script removes duplicate images, makes its own train/validation/test split, and uses the test set once at the end.

In [ ]:
!python training/train_mri.py

**5. Train the sleep staging model.** People, not nights, are split into train, validation and test.

In [ ]:
!python training/train_sleep.py

**6. Convert both models to ONNX**, which is what the app runs, check that the converted models give the same answers as PyTorch, and refit the check that warns about unfamiliar scans.

In [ ]:
!python training/export_onnx.py
!python training/fit_familiarity.py

**7. Download the result.** Unzip `neurova_retrained.zip` over your `neurova` folder on your laptop, replacing the old `models`, `results` and `samples` folders.

In [ ]:
!zip -qr neurova_retrained.zip models results samples
files.download("neurova_retrained.zip")